In [2]:
import os
import sys
sys.path.append("..")

from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv("../.env")
login(token=os.getenv("HF_TOKEN"), add_to_git_credential=False)

print("GOOGLE_API_KEY set:", bool(os.getenv("GOOGLE_API_KEY")))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GOOGLE_API_KEY set: True


In [3]:
from auto_pricer.car import Car

train, val, test = Car.from_hub("ShahidHKhan/cars_lite_raw")
print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")

sample = train[0]
print("\n--- car.full ---")
print(sample.full)

c:\Users\mrsha\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mrsha\.cache\huggingface\hub\datasets--ShahidHKhan--cars_lite_raw. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 1452/1452 [00:00<00:00, 82556.76 examples/s]


Train: 26,132, Val: 1,452, Test: 1,452

--- car.full ---
2008 Porsche 911 Turbo AWD Convertible
Body: Convertible | Mileage: 19,000 mi
Horsepower: 480.0 | Fuel: Gasoline
Transmission: Automatic
Accidents: False | Frame damaged: False
Description: 2008 Porsche 911 Turbo Black over an outrageous Special Natural Leather Cocoa interior with matching roof,Steering wheel floor mats and Rear Center console.   We see a lot of Porsche turbos and this one stands apart,phenomenal conditional and it drives beautifully!Also equipped with factory Sirius radio a car cover and clear expel paint protection on hood!NON-SMOKER,HEATED SEATS,Adaptive suspensio


In [4]:
import litellm
from litellm import completion

litellm.suppress_debug_info = True

SYSTEM_PROMPT = """Create a concise description of a used car listing. Respond only in this format.
Title: Rewritten short precise title (year make model trim)
Category: e.g. Sedan, SUV, Truck
Make: Brand name
Description: 1 sentence on condition/history
Details: 1 sentence on mileage, fuel type, transmission"""

def messages_for(text):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text}
    ]

response = completion(
    model="gemini/gemini-2.5-flash-lite",
    messages=messages_for(sample.full),
    timeout=15
)

print(response.choices[0].message.content)
print("\n--- usage ---")
print("input tokens:", response.usage.prompt_tokens)
print("output tokens:", response.usage.completion_tokens)
print("cost: $", response._hidden_params["response_cost"])

Title: 2008 Porsche 911 Turbo AWD Convertible
Category: Convertible
Make: Porsche
Description: Exceptional condition with a unique Special Natural Leather Cocoa interior and numerous factory options.
Details: 19,000 miles, gasoline, automatic transmission.

--- usage ---
input tokens: 219
output tokens: 58
cost: $ 4.51e-05


In [5]:
import time
import random

class Preprocessor:
    def __init__(self, model_name="gemini/gemini-2.5-flash-lite", max_retries=5):
        self.model_name = model_name
        self.max_retries = max_retries
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cost = 0

    def messages_for(self, text):
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text}
        ]

    def preprocess(self, car):
        messages = self.messages_for(car.full)
        for attempt in range(self.max_retries):
            try:
                response = completion(model=self.model_name, messages=messages, timeout=15)
                self.total_input_tokens += response.usage.prompt_tokens
                self.total_output_tokens += response.usage.completion_tokens
                self.total_cost += response._hidden_params["response_cost"]
                car.summary = response.choices[0].message.content
                return car
            except Exception:
                wait = min((2 ** attempt) + random.uniform(0, 1), 10)
                time.sleep(wait)
        return car  # leaves car.summary = None

preprocessor = Preprocessor()
print("Preprocessor ready")

Preprocessor ready


In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def run_preprocessing(preprocessor, cars, max_workers=10):
    pbar = tqdm(total=len(cars))
    failures = [0]

    def run_one(car):
        result = preprocessor.preprocess(car)
        if result.summary is None:
            failures[0] += 1
        pbar.update(1)
        pbar.set_postfix(failures=failures[0])
        return result

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(run_one, car) for car in cars]
        for f in as_completed(futures):
            f.result()
    pbar.close()
    print(f"Total cost: ${preprocessor.total_cost:.4f}")

test_batch = train[:25]
run_preprocessing(preprocessor, test_batch, max_workers=10)

# sanity check
failed = sum(c.summary is None for c in test_batch)
print(f"Failed: {failed}/25")
print("\n--- sample summary ---")
print(test_batch[0].summary)

100%|██████████| 25/25 [00:02<00:00, 11.05it/s, failures=0]

Total cost: $0.0013
Failed: 0/25

--- sample summary ---
Title: 2008 Porsche 911 Turbo Convertible
Category: Convertible
Make: Porsche
Description: Exceptionally maintained 2008 Porsche 911 Turbo convertible with a unique Cocoa leather interior and low mileage.
Details: 19,000 miles, gasoline, automatic transmission.


In [7]:
lite_cars = train + val + test
print(f"Total cars to process: {len(lite_cars):,}")

run_preprocessing(preprocessor, lite_cars, max_workers=10)

Total cars to process: 29,036


100%|██████████| 29036/29036 [28:44<00:00, 16.84it/s, failures=0]  

Total cost: $1.4291


In [8]:
# Verify no missing summaries
missing = sum(c.summary is None for c in lite_cars)
print(f"Missing summaries: {missing}")

# Spot check a couple more
for c in lite_cars[1:3]:
    print("\n---")
    print(c.summary)

Missing summaries: 0

---
Title: 2014 Volkswagen Passat SEL V6 Premium Sedan
Category: Sedan
Make: Volkswagen
Description: Well-maintained sedan with a 3.6L VR6 engine and luxurious black/gray leather interior.
Details: 87,618 miles, gasoline, 6-speed automatic transmission.

---
Title: 2009 Ford E-250 Cargo Van
Category: Van
Make: Ford
Description: Well-maintained cargo van with no reported accidents or frame damage.
Details: 165,200 miles, Flex Fuel, Automatic transmission.


In [9]:
Car.push_prompts_to_hub  # not yet — this is for Day 5 prompt/completion format

# This is the Day 2 push: strip `full`/`id`, keep everything else (including `summary`)
from datasets import Dataset, DatasetDict

def strip_for_processed(cars):
    return [
        {**c.model_dump(exclude={"full", "id"})}
        for c in cars
    ]

DatasetDict({
    "train": Dataset.from_list(strip_for_processed(train)),
    "validation": Dataset.from_list(strip_for_processed(val)),
    "test": Dataset.from_list(strip_for_processed(test)),
}).push_to_hub("ShahidHKhan/cars_lite_processed")

print("Pushed: ShahidHKhan/cars_lite_processed")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 17.43ba/s]
Processing Files (1 / 1): 100%|██████████| 2.38MB / 2.38MB,  188kB/s  
New Data Upload: 100%|██████████| 1.88MB / 1.88MB,  151kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.53s/ shards]
Setting num_proc from 1 back to 1 for the validation split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 66.94ba/s]
Processing Files (1 / 1): 100%|██████████|  157kB /  157kB, 15.2kB/s  
New Data Upload: 100%|██████████|  157kB /  157kB, 15.2kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.24 shards/s]
Setting num_proc from 1 back to 1 for the test split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:0

Pushed: ShahidHKhan/cars_lite_processed


In [1]:
%%writefile ../auto_pricer/evaluator.py
import re
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from itertools import accumulate
import math
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}

WORKERS = 5
DEFAULT_SIZE = 200


class Tester:
    def __init__(self, predictor, data, title=None, size=DEFAULT_SIZE, workers=WORKERS):
        self.predictor = predictor
        self.data = data
        self.title = title or self.make_title(predictor)
        self.size = size
        self.titles = []
        self.guesses = []
        self.truths = []
        self.errors = []
        self.colors = []
        self.workers = workers

    @staticmethod
    def make_title(predictor) -> str:
        return predictor.__name__.replace("__", ".").replace("_", " ").title().replace("Gpt", "GPT")

    @staticmethod
    def post_process(value):
        if isinstance(value, str):
            value = value.replace("$", "").replace(",", "")
            match = re.search(r"[-+]?\d*\.\d+|\d+", value)
            return float(match.group()) if match else 0
        else:
            return value

    def color_for(self, error, truth):
        if error < 40 or error / truth < 0.2:
            return "green"
        elif error < 80 or error / truth < 0.4:
            return "orange"
        else:
            return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        value = self.predictor(datapoint)
        guess = self.post_process(value)
        truth = datapoint.price
        error = abs(guess - truth)
        color = self.color_for(error, truth)
        title = datapoint.title if len(datapoint.title) <= 40 else datapoint.title[:40] + "..."
        return title, guess, truth, error, color

    def chart(self, title):
        df = pd.DataFrame({
            "truth": self.truths, "guess": self.guesses, "title": self.titles,
            "error": self.errors, "color": self.colors,
        })
        df["hover"] = [
            f"{t}\nGuess=${g:,.2f} Actual=${y:,.2f}"
            for t, g, y in zip(df["title"], df["guess"], df["truth"])
        ]
        max_val = float(max(df["truth"].max(), df["guess"].max()))
        fig = px.scatter(
            df, x="truth", y="guess", color="color",
            color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
            title=title, labels={"truth": "Actual Price", "guess": "Predicted Price"},
            width=1000, height=800,
        )
        for tr in fig.data:
            mask = df["color"] == tr.name
            tr.customdata = df.loc[mask, ["hover"]].to_numpy()
            tr.hovertemplate = "%{customdata[0]}<extra></extra>"
            tr.marker.update(size=6)
        fig.add_trace(go.Scatter(
            x=[0, max_val], y=[0, max_val], mode="lines",
            line=dict(width=2, dash="dash", color="deepskyblue"),
            name="y = x", hoverinfo="skip", showlegend=False,
        ))
        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])
        fig.update_layout(showlegend=False)
        fig.show()

    def error_trend_chart(self):
        n = len(self.errors)
        running_sums = list(accumulate(self.errors))
        x = list(range(1, n + 1))
        running_means = [s / i for s, i in zip(running_sums, x)]
        running_squares = list(accumulate(e * e for e in self.errors))
        running_stds = [
            math.sqrt((sq_sum / i) - (mean**2)) if i > 1 else 0
            for i, sq_sum, mean in zip(x, running_squares, running_means)
        ]
        ci = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]
        upper = [m + c for m, c in zip(running_means, ci)]
        lower = [m - c for m, c in zip(running_means, ci)]
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x + x[::-1], y=upper + lower[::-1], fill="toself",
            fillcolor="rgba(128,128,128,0.2)", line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip", showlegend=False, name="95% CI",
        ))
        fig.add_trace(go.Scatter(
            x=x, y=running_means, mode="lines", line=dict(width=3, color="firebrick"),
            name="Cumulative Avg Error", customdata=list(zip(ci,)),
            hovertemplate=("n=%{x}<br>Avg Error=$%{y:,.2f}<br>±95% CI=$%{customdata[0]:,.2f}<extra></extra>"),
        ))
        final_mean = running_means[-1]
        final_ci = ci[-1]
        title = f"{self.title} Error: ${final_mean:,.2f} ± ${final_ci:,.2f}"
        fig.update_layout(
            title=title, xaxis_title="Number of Datapoints", yaxis_title="Average Absolute Error ($)",
            width=1000, height=360, template="plotly_white", showlegend=False,
        )
        fig.show()

    def report(self):
        average_error = sum(self.errors) / self.size
        mse = mean_squared_error(self.truths, self.guesses)
        r2 = r2_score(self.truths, self.guesses) * 100
        title = f"{self.title} results<br><b>Error:</b> ${average_error:,.2f} <b>MSE:</b> {mse:,.0f} <b>r²:</b> {r2:.1f}%"
        self.error_trend_chart()
        self.chart(title)

    def run(self):
        with ThreadPoolExecutor(max_workers=self.workers) as ex:
            for title, guess, truth, error, color in tqdm(
                ex.map(self.run_datapoint, range(self.size)), total=self.size
            ):
                self.titles.append(title)
                self.guesses.append(guess)
                self.truths.append(truth)
                self.errors.append(error)
                self.colors.append(color)
                print(f"{COLOR_MAP[color]}${error:.0f} ", end="")
        self.report()


def evaluate(function, data, size=DEFAULT_SIZE, workers=WORKERS):
    Tester(function, data, size=size, workers=workers).run()

Overwriting ../auto_pricer/evaluator.py


In [8]:
from auto_pricer.evaluator import evaluate, Tester
print("evaluator imported OK")

ModuleNotFoundError: No module named 'auto_pricer'